# Topic: SQL: Conversion Funnel Analysis

## Definition (30-second explanation)
* A conversion funnel tracks the user journey through a predefined sequence of steps (e.g., page view -> add to cart -> checkout -> purchase).
* In SQL, it is used to calculate the number of unique users who reach each stage and identify where the highest drop-off occurs.

## Why Interviewers Ask This
* **Product Analytics Staple:** It is arguably the most common SQL task for product data scientists and growth analysts.
* **Complex Aggregation Skills:** It tests your ability to pivot rows into columns using conditional aggregation (`CASE WHEN`).
* **Business Acumen:** Tests if you can identify bottlenecks and suggest UX/product improvements based on drop-off data.

## Core Concepts
* **Conditional Aggregation:** Using `COUNT(DISTINCT CASE WHEN...)` to evaluate conditions row-by-row and aggregate the results into columns.
* **Drop-off Rate:** The percentage of users who fail to move from Step N to Step N+1.
* **Ordered vs. Unordered Funnels:** Unordered simply checks if the events happened. Ordered strict funnels verify that Event B happened *after* Event A.

## When to Use
* Identifying abandonment bottlenecks in e-commerce checkout flows.
* Tracking user onboarding completion in SaaS products.
* Measuring the effectiveness of marketing campaigns (Impression -> Click -> Sign-up).

## Advantages
* Highly customizable in SQL (can be grouped by device, geography, or time).
* Directly actionable for business stakeholders (pinpoints exactly where to focus engineering/design efforts).

## Limitations
* Standard conditional aggregation doesn't enforce strict chronological event ordering (users skipping steps might still be counted).
* Heavy queries: Running `COUNT(DISTINCT)` on massive event logs can be computationally expensive.

## Common Comparisons
* **`COUNT(event)` vs `COUNT(DISTINCT user_id)`:** `COUNT(event)` inflates metrics if a user refreshes the page or performs an action multiple times. Always use distinct users for funnels.
* **Wide vs. Long Format Output:** Funnels can be output as a single row (wide) using CTEs, or as multiple rows (long) using `UNION ALL`. 

## Common Interview Traps
* **Forgetting `DISTINCT`:** The #1 reason candidates fail this question is counting events instead of unique users.
* **Assuming strict order:** Not asking the interviewer if a user *must* complete Step 1 before Step 2.
* **Ignoring time bounds:** Failing to clarify if the funnel applies to a single session, a 24-hour window, or a lifetime.

## Python / SQL Syntax (if applicable)

    SELECT 
      COUNT(DISTINCT CASE WHEN event_name = 'page_view' THEN user_id END) AS step_1,
      COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN user_id END) AS step_2
    FROM events;

## Important Formula (if applicable)
* **Stage-to-Stage Conversion:** `(Stage 2 Users / Stage 1 Users) * 100`
* **Absolute Conversion:** `(Stage N Users / Stage 1 Users) * 100`
* **Drop-off Rate:** `100 - Stage-to-Stage Conversion`

## 45-Second Interview Answer
"Funnel analysis is essential for identifying where users drop off in a multi-step process. In SQL, I typically build this using conditional aggregation—specifically `COUNT(DISTINCT CASE WHEN event_name = 'x' THEN user_id END)`—to calculate the unique number of users at each stage. I wrap this in a CTE to compute conversion and drop-off rates. Before writing the query, I always clarify two things: whether the funnel needs strict chronological ordering, and what the time window for completion is."

## Example Questions:

### Q1. Find the step with the highest drop-off percentage
* **Ideal Interview Answer (MySQL):** 
    ```sql
    WITH stage_counts AS (
        SELECT 
            COUNT(DISTINCT CASE WHEN event_name = 'page_view' THEN user_id END) AS s1,
            COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart' THEN user_id END) AS s2,
            COUNT(DISTINCT CASE WHEN event_name = 'checkout' THEN user_id END) AS s3,
            COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN user_id END) AS s4
        FROM funnel_events
    ),
    dropoffs AS (
        SELECT 'page_view -> add_to_cart' AS funnel_step, 100.0 - (s2/s1*100) AS dropoff_pct FROM stage_counts
        UNION ALL
        SELECT 'add_to_cart -> checkout', 100.0 - (s3/s2*100) FROM stage_counts
        UNION ALL
        SELECT 'checkout -> purchase', 100.0 - (s4/s3*100) FROM stage_counts
    )
    SELECT funnel_step, dropoff_pct
    FROM dropoffs
    ORDER BY dropoff_pct DESC
    LIMIT 1;
    ```
* **Common Mistakes:** Trying to calculate drop-off percentages without first storing the base counts in a CTE, leading to messy, unreadable, and error-prone division logic.
* **Likely Follow-up:** How would you modify this query to handle cases where two steps have the exact same drop-off percentage? *(Answer: Use the `RANK()` or `DENSE_RANK()` window function instead of `ORDER BY ... LIMIT 1` to allow for ties).*

### Q2. Build a funnel broken down by device type (mobile vs desktop)
* **Ideal Interview Answer (MySQL):**
    ```sql
    SELECT 
        device_type,
        COUNT(DISTINCT CASE WHEN event_name = 'page_view' THEN user_id END) AS stage_1_view,
        COUNT(DISTINCT CASE WHEN event_name = 'add_to_cart' THEN user_id END) AS stage_2_cart,
        COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN user_id END) AS stage_3_purchase
    FROM funnel_events
    GROUP BY device_type;
    ```
* **Common Mistakes:** Filtering for a specific device in a `WHERE` clause instead of adding `device_type` to the `SELECT` and `GROUP BY` clauses.
* **Likely Follow-up:** If the `device_type` is stored in a separate `users` table, how would this query change?

### Q3. Calculate the average time users spend at each funnel stage
* **Ideal Interview Answer (MySQL):**
    ```sql
    WITH user_times AS (
        SELECT 
            user_id,
            event_name,
            event_time,
            LEAD(event_time) OVER (PARTITION BY user_id ORDER BY event_time) AS next_event_time
        FROM funnel_events
    )
    SELECT 
        event_name,
        AVG(TIMESTAMPDIFF(SECOND, event_time, next_event_time)) AS avg_time_seconds
    FROM user_times
    GROUP BY event_name;
    ```
* **Common Mistakes:** Trying to do this with simple `MIN()` and `MAX()` groupings, which fails if users loop through the funnel multiple times.
* **Likely Follow-up:** How does `LEAD()` handle the final 'purchase' event where there is no next step?

### Q4. Identify users who completed the funnel in under 10 minutes
* **Ideal Interview Answer (MySQL):**
    ```sql
    WITH user_funnel_times AS (
        SELECT 
            user_id,
            MIN(CASE WHEN event_name = 'page_view' THEN event_time END) AS first_view,
            MAX(CASE WHEN event_name = 'purchase' THEN event_time END) AS purchase_time
        FROM funnel_events
        GROUP BY user_id
    )
    SELECT user_id
    FROM user_funnel_times
    WHERE TIMESTAMPDIFF(MINUTE, first_view, purchase_time) < 10;
    ```
* **Common Mistakes:** Using an `INNER JOIN` on the table to itself four times, which scales terribly with large event logs.
* **Likely Follow-up:** What happens in this query if a user views a page on Monday, and then views a page and purchases on Tuesday? (Hint: MIN/MAX might stretch the window incorrectly).

### Q5. Build a weekly funnel showing conversion rate trends over 4 weeks
* **Ideal Interview Answer (MySQL):**
    ```sql
    SELECT 
        YEARWEEK(event_time) AS week_num,
        COUNT(DISTINCT CASE WHEN event_name = 'page_view' THEN user_id END) AS views,
        COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN user_id END) AS purchases,
        ROUND(COUNT(DISTINCT CASE WHEN event_name = 'purchase' THEN user_id END) / 
              COUNT(DISTINCT CASE WHEN event_name = 'page_view' THEN user_id END) * 100, 2) AS conversion_rate
    FROM funnel_events
    WHERE event_time >= DATE_SUB(CURDATE(), INTERVAL 4 WEEK)
    GROUP BY YEARWEEK(event_time)
    ORDER BY week_num;
    ```
* **Common Mistakes:** Grouping by just `WEEK()` without accounting for the year, which causes data collision if the dataset spans multiple years.
* **Likely Follow-up:** How would you handle a user who views a page on Sunday (Week 1) but purchases on Monday (Week 2)?

## Practice Questions:

### Q1: 
**Using the classicmodels schema, write a single MySQL query to create a 3-stage business funnel returning a single row. Calculate the absolute number of unique customers who:**

- Stage 1 (Registered): Exist in the customers table.

- Stage 2 (Placed Order): Have placed at least one order (exist in the orders table).

- Stage 3 (Successful Payment): Have made at least one payment (exist in the payments table).

*Note: Assume a standard relational left-join approach. You do not need to calculate percentages, just the distinct customer counts for these three stages.*

**Answer:**
```sql
SELECT
        COUNT(DISTINCT c.customerNumber) AS stage_1_registered,
        COUNT(DISTINCT o.customerNumber) AS stage_2_ordered,
        COUNT(DISTINCT p.customerNumber) AS stage_3_paid
    FROM customers c 
    LEFT JOIN orders o
        ON c.customerNumber = o.customerNumber
    LEFT JOIN payments p
        ON o.customerNumber = p.customerNumber;
```

*Explanation:* "I use a sequence of LEFT JOINs to retain the total base of customers. Because a single customer can have multiple orders and multiple payments, chaining these tables causes a Cartesian product (row fan-out). Using `COUNT(DISTINCT)` is absolutely critical here to ensure we are counting unique users at each stage and not artificially inflating the funnel."
* **Common Mistakes:** Using `COUNT(o.customerNumber)` without `DISTINCT`, which inflates the funnel numbers due to the one-to-many row multiplication. 
* **Likely Follow-up:** How would you modify this to only include orders placed in the year 2023? *(Answer: Move the date filter into the `ON` clause of the left join, NOT the `WHERE` clause, otherwise it converts the left join into an inner join and destroys the top of the funnel).*

### Q2: You are analyzing the funnel_events table from the previous notes: (user_id, event_name, event_time)
**Write a SQL query to calculate the number of unique users who completed a strict time-ordered 2-stage funnel.**

Specifically, count how many users had a 'page_view' and subsequently had an 'add_to_cart' event where the event_time of the cart addition was strictly after the event_time of their page view.

* **Answer:** 
```sql
    SELECT 
        COUNT(DISTINCT f1.user_id) AS completed_2_steps
    FROM funnel_events f1
    JOIN funnel_events f2 
        ON f1.user_id = f2.user_id
    WHERE f1.event_name = 'page_view'
      AND f2.event_name = 'add_to_cart'
      AND f2.event_time > f1.event_time;
```

*Explanation:* "To enforce strict chronological order between two specific steps, I use a self-join on the `user_id`. I filter the first table alias for the first event, the second alias for the second event, and explicitly require that the timestamp of the second event is strictly greater than the first."
* **Common Mistakes:** Using the wrong inequality sign (`f1.event_time > f2.event_time`), which looks for users traveling backward in time. Using a standard conditional `COUNT(DISTINCT CASE...)` which does not check the timestamps at all.
* **Likely Follow-up:** How would you write this if you needed to track a strict 4-step funnel? *(Answer: Self-joins get messy for 4+ steps. I would use Window Functions, specifically `LEAD()`, to pull the next event's name and time into the same row, or use a pattern matching function like `MATCH_RECOGNIZE` if the SQL dialect supports it).*